# 01 — Canon Overlap

Builds the book presence/absence matrix across the Protestant, Catholic, and Orthodox canons, then visualizes it as a bipartite overlap graph.

Data source: `data/metadata/canon_lists.json`, loaded via `src/data_loader.load_canon_list`. See `docs/historical_notes.md` for the case studies (2 Maccabees 12:38-46, 2 Kings 13:20-21) that motivate this analysis.

In [2]:
import sys
sys.path.append('../src')

from data_loader import load_canon_list
from align_corpus import align_books, unique_to_tradition
from graph_builder import build_canon_overlap_graph

import matplotlib.pyplot as plt
import networkx as nx


ModuleNotFoundError: No module named 'networkx'

## Load canon lists

In [ ]:
protestant_books = load_canon_list('protestant')
catholic_books = load_canon_list('catholic')
orthodox_books = load_canon_list('orthodox')

print(f'Protestant: {len(protestant_books)} books')
print(f'Catholic:   {len(catholic_books)} books')
print(f'Orthodox:   {len(orthodox_books)} books')


## Build the presence/absence matrix

In [ ]:
matrix = align_books(protestant_books, catholic_books, orthodox_books)

print(f'{len(matrix)} unique books across all three traditions')


## Books unique to each tradition

Orthodox-only books should include things like 1 Esdras, 3 Maccabees, Prayer of Manasseh, and Psalm 151. There should be no Protestant-only or Catholic-only books, since both are proper subsets of the Orthodox list in this dataset — worth revisiting once the Orthodox list is cross-checked against a specific jurisdiction (see the To Do in `canon_lists.md`).

In [ ]:
for tradition in ('protestant', 'catholic', 'orthodox'):
    unique = unique_to_tradition(matrix, tradition)
    print(f'{tradition} ({len(unique)} unique): {unique}')


## Spot-check the 2 Maccabees / 2 Kings case studies

Confirms the canon status described in `docs/historical_notes.md`: 2 Maccabees should be Catholic/Orthodox-only, 2 Kings should be present in all three.

In [ ]:
for book in ('2 Maccabees', '2 Kings'):
    print(book, matrix[book])


## Overlap graph

In [ ]:
G = build_canon_overlap_graph(matrix)

tradition_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'tradition']
book_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'book']

print(f'{len(tradition_nodes)} tradition nodes, {len(book_nodes)} book nodes, '
      f'{G.number_of_edges()} edges')


In [ ]:
pos = nx.spring_layout(G, seed=42, k=0.3)

fig, ax = plt.subplots(figsize=(14, 14))

nx.draw_networkx_nodes(G, pos, nodelist=tradition_nodes, node_color='tab:red',
                       node_size=800, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=book_nodes, node_color='tab:blue',
                       node_size=100, alpha=0.7, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.2, ax=ax)
nx.draw_networkx_labels(G, pos, labels={n: n for n in tradition_nodes},
                        font_size=14, font_weight='bold', ax=ax)

ax.set_title('Canon Overlap: Traditions and Books')
ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/figures/canon_overlap_graph.png', dpi=150)
plt.show()
